# Full-Volume HD95 GPU Pass

This notebook computes full-volume HD95 separately from the main benchmark. It reruns only the predictions needed for rows in a completed `per_case_results.jsonl`, groups work by case, reuses equivalent prompts, crops to the union foreground volume, and caches the GT distance transform once per case.

The output is a JSONL/CSV that can be merged back into benchmark results by `(model, prompt_mode, dataset_index)`.

In [ ]:
import os
import sys
import gc
import csv
import json
import time
import zipfile
import math
import glob
import pickle
import random
import inspect
import traceback
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from IPython.display import display

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import SimpleITK as sitk
except Exception:
    sitk = None

try:
    from scipy.ndimage import distance_transform_edt, binary_erosion
except Exception:
    distance_transform_edt = None
    binary_erosion = None

SEED = 42
TARGET_HW = 512
LOADERS_PKL = "/data/loaders32.pkl"
EXCLUDED_DATASET = "wanglab__CT_DeepLesion-MedSAM2"

# Full benchmark knobs. Start small for smoke/debug, then set MAX_TEST_CASES = None.
MAX_TEST_CASES = None
SMOKE_CASES_PER_DIM = 1
SMOKE_SCAN_MAX_CASES = 2048
PROMPT_MODES = ["box", "mask", "mixed"]
# Oxford/UofT validation notebooks use box prompts only. Keep unsupported mask/mixed
# runs off by default so summary tables do not score incompatible protocols as fair baselines.
EVALUATE_UNSUPPORTED_EXTERNAL_PROMPTS = False
EXTERNAL_BASELINE_PROMPT_MODES = ["box"]
# External Oxford/UofT adapters expect image tensors in [0,1]. RWKV loaders may return
# 0-255 2D images or z-scored 3D volumes, so normalize only for those baselines.
EXTERNAL_INPUT_PREPROCESS = "auto01"
MIXED_PROMPT_PROBS = {"box": 0.5, "mask": 0.5}
THRESHOLDS = np.linspace(0.05, 0.95, 19, dtype=np.float32)
PRIMARY_THRESHOLD = 0.5

# HD95 is very expensive on 3D volumes because it runs distance transforms over
# every slice and/or the full volume. Keep 2D HD95 for paper-style reporting,
# but skip 3D HD95 by default so volumetric inference can finish.
COMPUTE_HD95_2D = True
COMPUTE_HD95_2D_FOR_3D_CASES = False
COMPUTE_HD95_3D = False

# Resume from a previous benchmark JSONL. Completed rows are copied into the new
# run and skipped by (model, prompt_mode, dataset_index), preserving prior work.
RESUME_COMPLETED_ROWS = True
RESUME_FROM_RUN_DIR = "/data/rwkv_medsam2_model_comparisons/compare_oxford_metrics_20260526_212815"

# Full benchmark progress reporting. JSONL is flushed every row; CSV/status/TensorBoard are periodic.
BENCHMARK_STATUS_EVERY_SEC = 60
BENCHMARK_PARTIAL_CSV_EVERY_ROWS = 5000
BENCHMARK_TENSORBOARD = True
BENCHMARK_TB_TABLE_EVERY_ROWS = 5000
BENCHMARK_REUSE_EFFECTIVE_PROMPT_RESULTS = True
BENCHMARK_CLEAR_CUDA_CACHE_EVERY_RUN = False
BENCHMARK_RESET_CUDA_PEAK_EVERY_RUN = True

RUN_ROOT = Path("/data/rwkv_medsam2_model_comparisons")
RUN_NAME = time.strftime("full_volume_hd95_gpu_%Y%m%d_%H%M%S")
OUT_DIR = RUN_ROOT / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_ROOT_CANDIDATES = [
    os.getcwd(),
    "/RWKV-MedSAM2",
    "/data/jrbonill/RWKV-MedSAM2",
    "/data/jrbonill/rwkv_medsam2",
    "E:/dev/RWKV-MedSAM2",
]

OXFORD_ENV_BASE = "/data/jrbonill/oxford_medsam2_env"
OXFORD_REPO = f"{OXFORD_ENV_BASE}/Medical-SAM2"
OXFORD_SITE = f"{OXFORD_ENV_BASE}/site"
OXFORD_SAM_CKPT = f"{OXFORD_REPO}/checkpoints/sam2_hiera_tiny.pt"
OXFORD_MED_PRETRAIN = f"{OXFORD_REPO}/checkpoints/MedSAM2_pretrain.pth"

UOFT_MEDSAM2_BASE = "/data/jrbonill/medsam2_env/MedSAM2"
UOFT_MEDSAM2_SITE = "/data/jrbonill/medsam2_env/site"
UOFT_MEDSAM2_CKPT = f"{UOFT_MEDSAM2_BASE}/checkpoints/MedSAM2_latest.pt"
UOFT_MEDSAM2_CFG = "sam2.1_hiera_t512.yaml"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
print("OUT_DIR:", OUT_DIR)

HD95_SOURCE_RUN_DIR = Path("/data/rwkv_medsam2_model_comparisons/compare_oxford_metrics_20260526_212815")
HD95_SOURCE_CSV = HD95_SOURCE_RUN_DIR / "per_case_results.csv"
HD95_SOURCE_JSONL = HD95_SOURCE_RUN_DIR / "per_case_results.jsonl"
HD95_OUTPUT_JSONL = OUT_DIR / "full_volume_hd95_gpu.jsonl"
HD95_OUTPUT_CSV = OUT_DIR / "full_volume_hd95_gpu.csv"
HD95_OUTPUT_MERGE_CSV = OUT_DIR / "full_volume_hd95_gpu_merge_columns.csv"
HD95_MODEL_PROMPT_SUMMARY_CSV = OUT_DIR / "full_volume_hd95_model_prompt_summary.csv"
HD95_DATASET_MODEL_PROMPT_SUMMARY_CSV = OUT_DIR / "full_volume_hd95_dataset_model_prompt_summary.csv"
HD95_PAIRED_MODEL_DIFF_SUMMARY_CSV = OUT_DIR / "full_volume_hd95_paired_model_difference_summary.csv"
HD95_MANIFEST_JSON = OUT_DIR / "full_volume_hd95_manifest.json"
HD95_SUMMARY_MD = OUT_DIR / "full_volume_hd95_summary.md"
HD95_PACKET_ZIP = OUT_DIR / "full_volume_hd95_packet.zip"
HD95_ONLY_DIM = 3
HD95_DATASET_ALLOWLIST = []  # Fill with paper dataset names before a full run.
HD95_DATASET_INDEX_ALLOWLIST = []  # Optional exact dataset_index subset.
HD95_REQUIRE_DATASET_FILTER = True
HD95_MODELS = [
    "rwkv_medsam2_distill",
    "rwkv_medsam2_nodistill",
    "sam2_1_base",
    "oxford_medical_sam2",
    "uoft_medsam2",
]
HD95_PROMPT_MODES = ["box", "mask", "mixed"]
HD95_MAX_CASES = None
HD95_WRITE_EVERY_ROWS = 25
HD95_USE_GPU = True
HD95_THRESHOLD = PRIMARY_THRESHOLD
HD95_SOURCE_HD95_COLUMNS_ARE_IGNORED = True
print("HD95_SOURCE_CSV:", HD95_SOURCE_CSV)
print("HD95_SOURCE_JSONL fallback:", HD95_SOURCE_JSONL)
print("HD95_OUTPUT_JSONL:", HD95_OUTPUT_JSONL)


In [ ]:
def set_determinism(seed=SEED):
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

set_determinism(SEED)

warnings.filterwarnings(
    "ignore",
    message=".*Deterministic behavior was enabled.*",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    message=".*Memory Efficient attention.*",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    message=".*Flash attention.*",
    category=UserWarning,
)


def find_repo_root():
    for root in REPO_ROOT_CANDIDATES:
        root = os.path.abspath(str(root))
        if os.path.isfile(os.path.join(root, "rwkv_medsam2", "train_sam2.py")):
            return root
    raise FileNotFoundError(f"Could not find RWKV-MedSAM2 repo from: {REPO_ROOT_CANDIDATES}")

REPO_ROOT = find_repo_root()
EXT_ROOT = os.path.join(REPO_ROOT, "ext")
os.chdir(REPO_ROOT)
for p in [REPO_ROOT, EXT_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)
print("REPO_ROOT:", REPO_ROOT)

from hydra.core.global_hydra import GlobalHydra
from hydra import initialize_config_dir
from omegaconf import OmegaConf

from rwkv_medsam2.train_sam2 import load_config, get_data_loaders, build_student_predictor, collate_sam2_batch
from rwkv_medsam2.functions.func_metrics import dice_iou


def load_saved_loaders_robust(pkl_path):
    with open(pkl_path, "rb") as f:
        obj = pickle.load(f)
    if isinstance(obj, (tuple, list)) and len(obj) == 3:
        return obj[0], obj[1], obj[2]
    if isinstance(obj, dict):
        key_sets = [
            ("train_loader", "val_loader", "test_loader"),
            ("train_loaders", "val_loaders", "test_loaders"),
            ("train", "val", "test"),
        ]
        for keys in key_sets:
            if all(k in obj for k in keys):
                return obj[keys[0]], obj[keys[1]], obj[keys[2]]
        loader_like = [v for v in obj.values() if hasattr(v, "dataset") and hasattr(v, "__iter__")]
        if len(loader_like) >= 3:
            return loader_like[:3]
    raise RuntimeError(f"Unsupported loader pickle format: {pkl_path}")


def apply_dataset_exclusion(loader, excluded_dataset=EXCLUDED_DATASET):
    if loader is None or not hasattr(loader, "dataset") or not hasattr(loader.dataset, "sequences"):
        return {"before": None, "after": None, "removed": 0}
    before = len(loader.dataset.sequences)
    kept = [s for s in loader.dataset.sequences if str(s.get("dataset", "")).strip() != excluded_dataset]
    removed = before - len(kept)
    loader.dataset.sequences = kept
    if hasattr(loader.dataset, "entry_dims"):
        loader.dataset.entry_dims = [s.get("dim", 2) for s in kept]
    return {"before": before, "after": len(kept), "removed": removed}


def load_benchmark_loaders():
    if os.path.exists(LOADERS_PKL):
        print("Loading saved loaders:", LOADERS_PKL)
        train_loader, val_loader, test_loader = load_saved_loaders_robust(LOADERS_PKL)
    else:
        print("Saved loaders missing; rebuilding from VCR config.")
        cfg = load_config(os.path.join(REPO_ROOT, "ext", "sam2", "configs", "sam2.1", "sam2.1_vcr.yaml"))
        train_loader, val_loader, test_loader = get_data_loaders(cfg)
    for name, loader in [("train", train_loader), ("val", val_loader), ("test", test_loader)]:
        print(name, apply_dataset_exclusion(loader))
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = load_benchmark_loaders()
benchmark_loader = test_loader
print("Test batches:", len(benchmark_loader), flush=True)


def first_cases_by_dim(loader, n_per_dim=1, max_scan=SMOKE_SCAN_MAX_CASES):
    out = {2: [], 3: []}
    ds = loader.dataset

    # Fast metadata path: avoids loading full image volumes just to find smoke cases.
    dims = getattr(ds, "entry_dims", None)
    if dims is None and hasattr(ds, "sequences"):
        dims = [s.get("dim", 2) for s in ds.sequences]
    if dims is not None:
        for i, dim in enumerate(dims):
            dim = int(dim)
            if dim in out and len(out[dim]) < n_per_dim:
                out[dim].append(i)
            if all(len(v) >= n_per_dim for v in out.values()):
                print("Smoke scan used dataset metadata; no samples loaded.", flush=True)
                return out
        print("Smoke scan used dataset metadata but did not find every dim.", out, flush=True)
        return out

    scan_total = min(len(ds), int(max_scan)) if max_scan is not None else len(ds)
    print(f"Scanning up to {scan_total} samples for smoke-case dims...", flush=True)
    for i in tqdm(range(scan_total), desc="Find smoke cases", unit="case"):
        try:
            item = ds[i]
            dim = int(item.get("dim", 2))
            if dim in out and len(out[dim]) < n_per_dim:
                out[dim].append(i)
            if all(len(v) >= n_per_dim for v in out.values()):
                break
        except Exception as e:
            if i < 5:
                print(f"Skipping smoke candidate {i}: {type(e).__name__}: {e}", flush=True)
            continue
    return out

print("Smoke dataset indices by dim:", first_cases_by_dim(benchmark_loader, SMOKE_CASES_PER_DIM), flush=True)


In [ ]:
def sigmoid_np(x):
    x = np.asarray(x, dtype=np.float32)
    x = np.clip(x, -30.0, 30.0)
    return 1.0 / (1.0 + np.exp(-x))


def ensure_thw(x):
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    x = np.asarray(x)
    while x.ndim > 3 and 1 in x.shape:
        x = np.squeeze(x)
    if x.ndim == 2:
        x = x[None, ...]
    if x.ndim != 3:
        raise ValueError(f"Expected [T,H,W] or [H,W], got shape {x.shape}")
    return x


def confusion_counts(pred, gt):
    p = np.asarray(pred).astype(bool)
    g = np.asarray(gt).astype(bool)
    tp = int(np.logical_and(p, g).sum())
    fp = int(np.logical_and(p, ~g).sum())
    fn = int(np.logical_and(~p, g).sum())
    tn = int(np.logical_and(~p, ~g).sum())
    return tp, fp, fn, tn


def safe_div(num, den, default=float("nan")):
    den = float(den)
    return float(num) / den if den != 0 else default


def threshold_suffix(thr):
    return f"{float(thr):.2f}".replace(".", "p")


def threshold_metric_key(metric, thr):
    return f"{metric}_thr_{threshold_suffix(thr)}"


def metric_at_threshold(row, metric, thr):
    return row.get(threshold_metric_key(metric, thr), float("nan"))


def hd95_binary(pred, gt, spacing=None):
    pred = np.asarray(pred).astype(bool)
    gt = np.asarray(gt).astype(bool)
    if pred.shape != gt.shape:
        raise ValueError(f"HD95 shape mismatch: {pred.shape} vs {gt.shape}")
    if not pred.any() and not gt.any():
        return 0.0
    if pred.any() != gt.any():
        return float("nan")
    if distance_transform_edt is None or binary_erosion is None:
        return float("nan")
    if spacing is None:
        spacing = (1.0,) * pred.ndim
    pred_border = np.logical_xor(pred, binary_erosion(pred))
    gt_border = np.logical_xor(gt, binary_erosion(gt))
    if not pred_border.any() or not gt_border.any():
        return 0.0 if np.array_equal(pred, gt) else float("nan")
    dt_pred = distance_transform_edt(~pred_border, sampling=spacing)
    dt_gt = distance_transform_edt(~gt_border, sampling=spacing)
    distances = np.concatenate([dt_gt[pred_border], dt_pred[gt_border]]).astype(np.float64)
    if distances.size == 0:
        return float("nan")
    return float(np.percentile(distances, 95))


def binary_dice_iou_np(pred, gt):
    tp, fp, fn, tn = confusion_counts(pred, gt)
    both_empty = (np.asarray(pred).sum() == 0 and np.asarray(gt).sum() == 0)
    dice = safe_div(2 * tp, 2 * tp + fp + fn, default=1.0 if both_empty else 0.0)
    iou = safe_div(tp, tp + fp + fn, default=1.0 if both_empty else 0.0)
    return float(dice), float(iou)


def training_style_metrics_from_logits(logits, gt, thresholds=THRESHOLDS):
    # Mirrors train_sam2 validation: non-empty GT frames only for slice metrics,
    # logit threshold 0.0 for @0.5 probability, and separate volume scores.
    nonempty = gt.reshape(gt.shape[0], -1).sum(axis=1) > 0
    out = {
        "train_nonempty_frames": int(nonempty.sum()),
        "train_total_frames": int(gt.shape[0]),
    }
    if not np.any(nonempty):
        out.update({
            "train_slice_dice@0.5": float("nan"),
            "train_slice_iou@0.5": float("nan"),
            "train_slice_dice_best": float("nan"),
            "train_slice_iou_best": float("nan"),
            "train_best_prob_threshold": float("nan"),
            "train_volume_dice@0.5": float("nan"),
            "train_volume_iou@0.5": float("nan"),
            "train_volume_dice_best": float("nan"),
            "train_volume_iou_best": float("nan"),
        })
        return out

    logits_ne = logits[nonempty]
    gt_ne = gt[nonempty]
    pred05 = (logits_ne > 0.0).astype(np.uint8)
    slice_d, slice_i = [], []
    for t in range(gt_ne.shape[0]):
        d, i = binary_dice_iou_np(pred05[t], gt_ne[t])
        slice_d.append(d)
        slice_i.append(i)

    best_d = -1.0
    best_i = -1.0
    best_thr = float(thresholds[0])
    for thr_prob in thresholds:
        thr_prob = float(thr_prob)
        thr_logit = float(np.log(thr_prob / (1.0 - thr_prob)))
        pred = (logits_ne > thr_logit).astype(np.uint8)
        d_list, i_list = [], []
        for t in range(gt_ne.shape[0]):
            d, i = binary_dice_iou_np(pred[t], gt_ne[t])
            d_list.append(d)
            i_list.append(i)
        d_m = float(np.mean(d_list))
        i_m = float(np.mean(i_list))
        if d_m > best_d:
            best_d = d_m
            best_i = i_m
            best_thr = thr_prob

    best_logit = float(np.log(best_thr / (1.0 - best_thr)))
    pred_best = (logits_ne > best_logit).astype(np.uint8)
    vol_d05, vol_i05 = binary_dice_iou_np(pred05, gt_ne)
    vol_db, vol_ib = binary_dice_iou_np(pred_best, gt_ne)

    out.update({
        "train_slice_dice@0.5": float(np.mean(slice_d)),
        "train_slice_iou@0.5": float(np.mean(slice_i)),
        "train_slice_dice_best": float(best_d),
        "train_slice_iou_best": float(best_i),
        "train_best_prob_threshold": float(best_thr),
        "train_volume_dice@0.5": float(vol_d05),
        "train_volume_iou@0.5": float(vol_i05),
        "train_volume_dice_best": float(vol_db),
        "train_volume_iou_best": float(vol_ib),
    })
    return out


def segmentation_metrics_from_logits(
    logits_thw,
    gt_thw,
    thresholds=THRESHOLDS,
    compute_hd95_2d=COMPUTE_HD95_2D,
    compute_hd95_2d_for_3d=COMPUTE_HD95_2D_FOR_3D_CASES,
    compute_hd95_3d=COMPUTE_HD95_3D,
):
    metric_t0 = time.perf_counter()
    logits = ensure_thw(logits_thw).astype(np.float32)
    gt = (ensure_thw(gt_thw) > 0).astype(np.uint8)
    if logits.shape != gt.shape:
        raise ValueError(f"Metric shape mismatch: logits {logits.shape} vs gt {gt.shape}")
    probs = sigmoid_np(logits)

    pred05 = (probs >= PRIMARY_THRESHOLD).astype(np.uint8)
    tp, fp, fn, tn = confusion_counts(pred05, gt)
    dice05, iou05 = binary_dice_iou_np(pred05, gt)

    threshold_metrics = {}
    dice_vals, iou_vals = [], []
    best_dice_score = -1.0
    best_dice_iou = float("nan")
    best_dice_thr = float(PRIMARY_THRESHOLD)
    for thr in thresholds:
        thr = float(thr)
        pred = (probs >= thr).astype(np.uint8)
        d, i = binary_dice_iou_np(pred, gt)
        threshold_metrics[threshold_metric_key("dice", thr)] = float(d)
        threshold_metrics[threshold_metric_key("iou", thr)] = float(i)
        dice_vals.append(d)
        iou_vals.append(i)
        if d > best_dice_score:
            best_dice_score = float(d)
            best_dice_iou = float(i)
            best_dice_thr = float(thr)

    hd95_2d_sec = 0.0
    hd95_3d_sec = 0.0
    hd95_value = float("nan")
    hd95_3d_value = float("nan")
    should_compute_2d_hd95 = bool(compute_hd95_2d) and (gt.shape[0] == 1 or bool(compute_hd95_2d_for_3d))
    if should_compute_2d_hd95:
        hd_t0 = time.perf_counter()
        hd95_2d = [hd95_binary(pred05[t], gt[t]) for t in range(gt.shape[0])]
        hd95_2d_sec = time.perf_counter() - hd_t0
        finite_hd = [x for x in hd95_2d if not (math.isnan(x) or math.isinf(x))]
        hd95_value = float(np.mean(finite_hd)) if finite_hd else float("nan")

    if bool(compute_hd95_3d) and gt.shape[0] > 1:
        hd_t0 = time.perf_counter()
        hd95_3d_value = hd95_binary(pred05, gt)
        hd95_3d_sec = time.perf_counter() - hd_t0

    metrics = {
        "dice": float(dice05),
        "iou": float(iou05),
        "hd95": hd95_value,
        "hd95_3d": hd95_3d_value,
        "hd95_2d_computed": float(should_compute_2d_hd95),
        "hd95_3d_computed": float(bool(compute_hd95_3d) and gt.shape[0] > 1),
        "hd95_2d_time_sec": float(hd95_2d_sec),
        "hd95_3d_time_sec": float(hd95_3d_sec),
        "dice_avg_threshold": float(np.mean(dice_vals)),
        "iou_avg_threshold": float(np.mean(iou_vals)),
        "dice_best_threshold": float(best_dice_thr),
        "dice_best": float(best_dice_score),
        "iou_at_dice_best": float(best_dice_iou),
        "logit_min": float(np.min(logits)),
        "logit_mean": float(np.mean(logits)),
        "logit_max": float(np.max(logits)),
        "prob_min": float(np.min(probs)),
        "prob_mean": float(np.mean(probs)),
        "prob_max": float(np.max(probs)),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "false_negative_rate": safe_div(fn, tp + fn, default=0.0),
        "false_positive_rate": safe_div(fp, fp + tn, default=0.0),
        "precision_ppv": safe_div(tp, tp + fp, default=1.0 if pred05.sum() == 0 and gt.sum() == 0 else 0.0),
        "recall_sensitivity": safe_div(tp, tp + fn, default=1.0 if gt.sum() == 0 else 0.0),
        "specificity": safe_div(tn, tn + fp, default=0.0),
        "volumetric_similarity": 1.0 - safe_div(abs(int(pred05.sum()) - int(gt.sum())), int(pred05.sum()) + int(gt.sum()), default=0.0),
        "pred_fg_frac": float(pred05.mean()),
        "gt_fg_frac": float(gt.mean()),
        "empty_pred": float(pred05.sum() == 0),
        "empty_gt": float(gt.sum() == 0),
        "missed_object": float((gt.sum() > 0) and (pred05.sum() == 0)),
        "hallucination": float((gt.sum() == 0) and (pred05.sum() > 0)),
        "n_frames": int(gt.shape[0]),
    }
    metrics.update(threshold_metrics)
    metrics.update(training_style_metrics_from_logits(logits, gt, thresholds=thresholds))
    metrics["metric_time_sec"] = float(time.perf_counter() - metric_t0)
    return metrics

def threshold_score_is_better(score, best_score, thr, best_thr):
    if math.isnan(score):
        return False
    if math.isnan(best_score) or score > best_score:
        return True
    if np.isclose(score, best_score):
        return abs(float(thr) - PRIMARY_THRESHOLD) < abs(float(best_thr) - PRIMARY_THRESHOLD)
    return False


def add_best_threshold_columns(row, thresholds=THRESHOLDS):
    best_dice_score, best_dice_thr = float("nan"), float(PRIMARY_THRESHOLD)
    best_iou_score, best_iou_thr = float("nan"), float(PRIMARY_THRESHOLD)
    for thr in thresholds:
        thr = float(thr)
        dice_score = float(row.get(threshold_metric_key("dice", thr), float("nan")))
        iou_score = float(row.get(threshold_metric_key("iou", thr), float("nan")))
        if threshold_score_is_better(dice_score, best_dice_score, thr, best_dice_thr):
            best_dice_score, best_dice_thr = dice_score, thr
        if threshold_score_is_better(iou_score, best_iou_score, thr, best_iou_thr):
            best_iou_score, best_iou_thr = iou_score, thr

    row["best_dice_threshold"] = float(best_dice_thr)
    row["dice_at_best_dice_threshold"] = float(best_dice_score)
    row["iou_at_best_dice_threshold"] = float(row.get(threshold_metric_key("iou", best_dice_thr), float("nan")))
    row["best_iou_threshold"] = float(best_iou_thr)
    row["iou_at_best_iou_threshold"] = float(best_iou_score)
    row["dice_at_best_iou_threshold"] = float(row.get(threshold_metric_key("dice", best_iou_thr), float("nan")))
    return row


def aggregate_records(records, group_keys):
    buckets = defaultdict(list)
    for r in records:
        key = tuple(r.get(k, "") for k in group_keys)
        buckets[key].append(r)
    rows = []
    numeric_skip = {"tp", "fp", "fn", "tn"}
    for key, vals in buckets.items():
        row = {k: v for k, v in zip(group_keys, key)}
        row["n_cases"] = len(vals)
        metric_keys = sorted({mk for v in vals for mk in v.keys() if isinstance(v.get(mk), (int, float, np.integer, np.floating))})
        for mk in metric_keys:
            xs = [float(v[mk]) for v in vals if mk in v and isinstance(v[mk], (int, float, np.integer, np.floating)) and not math.isnan(float(v[mk]))]
            if not xs:
                continue
            if mk in numeric_skip:
                row[mk] = float(np.sum(xs))
            else:
                row[mk] = float(np.mean(xs))
        add_best_threshold_columns(row)
        rows.append(row)
    return rows


def threshold_sweep_records(records, group_keys, thresholds=THRESHOLDS):
    buckets = defaultdict(list)
    for r in records:
        key = tuple(r.get(k, "") for k in group_keys)
        buckets[key].append(r)

    rows = []
    for key, vals in buckets.items():
        base = {k: v for k, v in zip(group_keys, key)}
        for thr in thresholds:
            thr = float(thr)
            dice_key = threshold_metric_key("dice", thr)
            iou_key = threshold_metric_key("iou", thr)
            dice_vals = [float(v[dice_key]) for v in vals if dice_key in v and not math.isnan(float(v[dice_key]))]
            iou_vals = [float(v[iou_key]) for v in vals if iou_key in v and not math.isnan(float(v[iou_key]))]
            rows.append({
                **base,
                "threshold": thr,
                "n_cases": len(vals),
                "dice": float(np.mean(dice_vals)) if dice_vals else float("nan"),
                "iou": float(np.mean(iou_vals)) if iou_vals else float("nan"),
            })
    return rows


def write_table(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if pd is not None:
        pd.DataFrame(rows).to_csv(path, index=False)
    else:
        keys = sorted({k for r in rows for k in r.keys()})
        with open(path, "w", newline="") as f:
            if keys:
                w = csv.DictWriter(f, fieldnames=keys)
                w.writeheader()
                w.writerows(rows)
            else:
                f.write("")
    print("wrote", path)


In [ ]:
def bbox_from_mask_np(mask_hw, pad=5):
    mask = np.asarray(mask_hw) > 0
    ys, xs = np.where(mask)
    H, W = mask.shape[-2:]
    if len(xs) == 0:
        return np.array([0, 0, W - 1, H - 1], dtype=np.float32)
    x0 = max(0, int(xs.min()) - pad)
    y0 = max(0, int(ys.min()) - pad)
    x1 = min(W - 1, int(xs.max()) + pad)
    y1 = min(H - 1, int(ys.max()) + pad)
    return np.array([x0, y0, x1, y1], dtype=np.float32)


def ensure_tchw(tensor):
    x = tensor.detach().float() if torch.is_tensor(tensor) else torch.as_tensor(tensor).float()
    if x.ndim == 2:
        x = x.unsqueeze(0).unsqueeze(0)
    elif x.ndim == 3:
        # Dataset samples are normally [T,H,W] for grayscale volumes or [C,H,W] for a single image.
        x = x.unsqueeze(1) if x.shape[0] != 3 else x.unsqueeze(0)
    elif x.ndim != 4:
        raise ValueError(f"Expected image/mask with 2-4 dims, got shape {tuple(x.shape)}")
    return x


def ensure_image_t3hw(image):
    x = ensure_tchw(image)
    if x.shape[1] == 1:
        x = x.repeat(1, 3, 1, 1)
    elif x.shape[1] > 3:
        x = x[:, :3]
    if x.shape[1] != 3:
        raise ValueError(f"Expected image channels to resolve to 3, got shape {tuple(x.shape)}")
    return x.contiguous()


def sample_to_case(sample, dataset_index=None):
    image_t = ensure_image_t3hw(sample["image"])
    mask_t = ensure_tchw(sample["mask"])
    T = int(image_t.shape[0])
    if mask_t.shape[0] != T and mask_t.shape[0] == 1:
        mask_t = mask_t.repeat(T, 1, 1, 1)
    if mask_t.shape[1] != 1:
        mask_t = mask_t[:, :1]
    valid = sample.get("valid", torch.ones(T, dtype=torch.bool))
    if torch.is_tensor(valid):
        valid_t = valid.detach().bool().flatten()
    else:
        valid_t = torch.as_tensor(valid).bool().flatten()
    valid_t = valid_t[:T]
    gt_thw = mask_t[:, 0].detach().cpu().numpy() > 0.5
    boxes = np.stack([bbox_from_mask_np(gt_thw[t]) for t in range(T)], axis=0)
    meta = {
        "dataset_index": dataset_index,
        "dataset": sample.get("dataset", "unknown"),
        "subdataset": sample.get("subdataset", "default"),
        "task_id": sample.get("task_id", sample.get("task_label", "unknown")),
        "task_label": sample.get("task_label", sample.get("task_id", "unknown")),
        "modality": sample.get("modality", "unknown"),
        "dim": int(sample.get("dim", 2)),
        "seq_idx": sample.get("seq_idx", dataset_index),
    }
    return {
        "image": image_t,
        "mask": mask_t,
        "valid": valid_t,
        "gt_thw": gt_thw,
        "boxes": boxes,
        "val_prompt_cache": sample.get("val_prompt_cache", None),
        "meta": meta,
    }


def tensor_to_numpy_or_none(x):
    if x is None:
        return None
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def cached_prompt_plan_for_case(case, prompt_mode):
    cache = case.get("val_prompt_cache", None)
    if not isinstance(cache, dict):
        return None
    mode = str(prompt_mode).strip().lower()
    if mode in ("bbox", "boxes"):
        mode = "box"
    if mode not in ("box", "mask"):
        return None
    payload = cache.get(mode, None)
    if not isinstance(payload, dict):
        return None
    T = int(case["image"].shape[0])
    chosen = [int(x) for x in cache.get("chosen_frames", []) if 0 <= int(x) < T]
    if not chosen:
        return None

    boxes = case["boxes"].copy().astype(np.float32)
    mask_prompts = np.zeros_like(case["gt_thw"], dtype=np.float32)

    if mode == "box":
        cached_boxes = payload.get("bbox", [])
        has_any = False
        for t in chosen:
            bb = cached_boxes[t] if t < len(cached_boxes) else None
            bb_np = tensor_to_numpy_or_none(bb)
            if bb_np is None or bb_np.size == 0:
                continue
            boxes[t] = np.asarray(bb_np, dtype=np.float32).reshape(-1, 4)[0]
            has_any = True
        if not has_any:
            return None
    elif mode == "mask":
        cached_masks = payload.get("m_prompt", [])
        has_any = False
        for t in chosen:
            mm = cached_masks[t] if t < len(cached_masks) else None
            mm_np = tensor_to_numpy_or_none(mm)
            if mm_np is None or mm_np.size == 0:
                continue
            mask_prompts[t] = np.squeeze(mm_np).astype(np.float32)
            has_any = True
        if not has_any:
            return None

    return {
        "requested_prompt_mode": prompt_mode,
        "effective_prompt_mode": mode,
        "prompt_frames": chosen,
        "boxes": boxes,
        "mask_prompts": mask_prompts,
        "prompt_source": "val_prompt_cache",
    }


def prompt_plan_for_case(case, prompt_mode, seed=SEED):
    cached = cached_prompt_plan_for_case(case, prompt_mode)
    if cached is not None:
        return cached
    T = int(case["image"].shape[0])
    gt = case["gt_thw"]
    valid = case["valid"].detach().cpu().numpy().astype(bool)
    nonempty = gt.reshape(T, -1).sum(axis=1) > 0
    eligible = np.where(valid & nonempty)[0]
    if len(eligible) == 0:
        eligible = np.where(valid)[0]
    if len(eligible) == 0:
        eligible = np.arange(T)
    # For 3D, prompt one representative foreground frame; for 2D T is normally one.
    areas = gt.reshape(T, -1).sum(axis=1)
    prompt_frame = int(eligible[np.argmax(areas[eligible])]) if len(eligible) else 0
    rng = np.random.RandomState((int(seed) * 1000003 + int(case["meta"].get("dataset_index") or 0) * 97) % (2**32 - 1))
    if prompt_mode == "mixed":
        selected = rng.choice(["box", "mask"], p=[MIXED_PROMPT_PROBS["box"], MIXED_PROMPT_PROBS["mask"]]).item()
        cached_selected = cached_prompt_plan_for_case(case, selected)
        if cached_selected is not None:
            cached_selected["requested_prompt_mode"] = prompt_mode
            return cached_selected
    elif prompt_mode in ("box", "mask"):
        selected = prompt_mode
    else:
        raise ValueError(f"Unsupported prompt mode {prompt_mode}; clicks are intentionally disabled.")
    return {
        "requested_prompt_mode": prompt_mode,
        "effective_prompt_mode": selected,
        "prompt_frames": [prompt_frame],
        "boxes": case["boxes"].astype(np.float32),
        "mask_prompts": case["gt_thw"].astype(np.float32),
        "prompt_source": "generated",
    }



def case_input_diagnostics(case, prompt_plan=None):
    img = case["image"].detach().cpu().float().numpy()
    gt = np.asarray(case["gt_thw"]).astype(bool)
    valid = case["valid"].detach().cpu().numpy().astype(bool)
    out = {
        "image_min": float(np.nanmin(img)),
        "image_mean": float(np.nanmean(img)),
        "image_std": float(np.nanstd(img)),
        "image_max": float(np.nanmax(img)),
        "valid_frames": int(valid.sum()),
        "nonempty_frames": int((gt.reshape(gt.shape[0], -1).sum(axis=1) > 0).sum()),
    }
    if prompt_plan is not None:
        frames = [int(t) for t in prompt_plan.get("prompt_frames", []) if 0 <= int(t) < gt.shape[0]]
        boxes = np.asarray(prompt_plan.get("boxes", []), dtype=np.float32)
        if frames and boxes.size:
            areas = []
            gt_areas = []
            for t in frames:
                x0, y0, x1, y1 = boxes[t].reshape(-1)[:4]
                areas.append(max(0.0, float(x1 - x0 + 1.0)) * max(0.0, float(y1 - y0 + 1.0)) / float(TARGET_HW * TARGET_HW))
                gt_areas.append(float(gt[t].mean()))
            out.update({
                "prompt_frame_count": int(len(frames)),
                "prompt_box_area_frac_mean": float(np.mean(areas)),
                "prompt_gt_area_frac_mean": float(np.mean(gt_areas)),
            })
    return out


In [ ]:
@dataclass
class ModelEntry:
    name: str
    family: str
    config_path: str = ""
    checkpoint_dir: str = ""
    checkpoint_path: str = ""
    checkpoint_policy: str = "best_or_latest"
    builder: str = ""
    enabled: bool = True
    status: str = "unknown"
    notes: str = ""


def default_model_registry():
    return [
        ModelEntry(
            name="rwkv_medsam2_distill",
            family="rwkv",
            config_path=os.path.join(REPO_ROOT, "ext/sam2/configs/sam2.1/sam2.1_vcr.yaml"),
            checkpoint_dir=os.path.join(REPO_ROOT, "checkpoints", "sam2.1", "sam2.1_vcr"),
            builder="rwkv_student",
            notes="RWKV-MedSAM2 with distillation."
        ),
        ModelEntry(
            name="rwkv_medsam2_nodistill",
            family="rwkv",
            config_path=os.path.join(REPO_ROOT, "ext/sam2/configs/sam2.1/sam2.1_vcr_nodistill.yaml"),
            checkpoint_dir=os.path.join(REPO_ROOT, "checkpoints", "sam2.1", "sam2.1_vcr_nodistill"),
            builder="rwkv_student",
            notes="Pending remote training checkpoint."
        ),
        ModelEntry(
            name="sam2_1_base",
            family="rwkv_base_config",
            config_path=os.path.join(REPO_ROOT, "ext/sam2/configs/sam2.1/sam2.1_hiera_t512.yaml"),
            checkpoint_dir=os.path.join(REPO_ROOT, "checkpoints", "sam2.1", "sam2.1_hiera_t512_base"),
            builder="sam21_base_student",
            notes="Pending remote training checkpoint; config path follows training_sam2.1_base.ipynb."
        ),
        ModelEntry(
            name="oxford_medical_sam2",
            family="oxford",
            config_path="sam2_hiera_t",
            checkpoint_path=OXFORD_MED_PRETRAIN,
            checkpoint_dir=os.path.dirname(OXFORD_MED_PRETRAIN),
            builder="oxford_video_predictor",
            notes="Uses Oxford-Refine_v9.ipynb loading pattern."
        ),
        ModelEntry(
            name="uoft_medsam2",
            family="uoft",
            config_path=UOFT_MEDSAM2_CFG,
            checkpoint_path=UOFT_MEDSAM2_CKPT,
            checkpoint_dir=os.path.dirname(UOFT_MEDSAM2_CKPT),
            builder="uoft_medsam2_build_sam2",
            notes="Uses MedSAM2-Setup.ipynb and MedSAM2-Refine_v2.ipynb loading pattern."
        ),
    ]

MODEL_REGISTRY = default_model_registry()
for entry in MODEL_REGISTRY:
    print(asdict(entry))


def checkpoint_candidates_from_dir(checkpoint_dir):
    candidates = []
    for name in ["best.pth", "best_val.pth", "latest.pth"]:
        p = os.path.join(checkpoint_dir, name)
        if os.path.isfile(p):
            candidates.append(p)
    candidates.extend(sorted(glob.glob(os.path.join(checkpoint_dir, "epoch_*.pth"))))
    return candidates


def select_checkpoint_candidate(candidates, policy="best_or_latest"):
    candidates = [p for p in candidates if p and os.path.isfile(p)]
    if not candidates:
        return None
    if policy == "latest":
        return max(candidates, key=os.path.getmtime)
    for p in candidates:
        if os.path.basename(p) in ("best.pth", "best_val.pth"):
            return p
    return max(candidates, key=os.path.getmtime)


def checkpoint_search_names(entry: ModelEntry):
    raw = os.path.basename(str(entry.checkpoint_dir or entry.checkpoint_path)).rstrip(os.sep)
    names = {raw}
    if raw.endswith(".pth"):
        names.add(raw[:-4])
    else:
        names.add(raw + ".pth")
    if entry.name == "rwkv_medsam2_distill":
        names.update(["sam2.1_vcr", "sam2.1_vcr.pth"])
    elif entry.name == "rwkv_medsam2_nodistill":
        names.update(["sam2.1_vcr_nodistill", "sam2.1_vcr_nodistill.pth"])
    elif entry.name == "sam2_1_base":
        names.update(["sam2.1_hiera_t512_base", "sam2.1_hiera_t512_base.pth"])
    return [n for n in names if n]


def pick_checkpoint(entry: ModelEntry):
    if entry.checkpoint_path and os.path.isfile(entry.checkpoint_path):
        return entry.checkpoint_path
    if entry.checkpoint_dir and os.path.isfile(entry.checkpoint_dir):
        return entry.checkpoint_dir
    if entry.checkpoint_dir and os.path.isdir(entry.checkpoint_dir):
        found = select_checkpoint_candidate(checkpoint_candidates_from_dir(entry.checkpoint_dir), entry.checkpoint_policy)
        if found:
            return found

    ckpt_root = os.path.join(REPO_ROOT, "checkpoints")
    if os.path.isdir(ckpt_root):
        search_dirs = [ckpt_root, os.path.join(ckpt_root, "sam2.1")]
        for base in search_dirs:
            for name in checkpoint_search_names(entry):
                candidate = os.path.join(base, name)
                if os.path.isfile(candidate):
                    return candidate
                if os.path.isdir(candidate):
                    found = select_checkpoint_candidate(checkpoint_candidates_from_dir(candidate), entry.checkpoint_policy)
                    if found:
                        return found
        for name in checkpoint_search_names(entry):
            matches = glob.glob(os.path.join(ckpt_root, "**", name), recursive=True)
            for match in sorted(matches):
                if os.path.isfile(match):
                    return match
                if os.path.isdir(match):
                    found = select_checkpoint_candidate(checkpoint_candidates_from_dir(match), entry.checkpoint_policy)
                    if found:
                        return found
    return None


def checkpoint_debug(entry: ModelEntry):
    paths = [entry.checkpoint_path, entry.checkpoint_dir]
    rows = []
    for path in paths:
        if path:
            rows.append(f"{path} file={os.path.isfile(path)} dir={os.path.isdir(path)}")
    ckpt_root = os.path.join(REPO_ROOT, "checkpoints")
    rows.append(f"checkpoint_search_names={checkpoint_search_names(entry)}")
    rows.append(f"checkpoint_root={ckpt_root} dir={os.path.isdir(ckpt_root)}")
    return "; ".join(rows)


def unwrap_state_dict(obj):
    if isinstance(obj, dict):
        for key in ["model", "state_dict", "model_state_dict", "net", "network"]:
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
    return obj


def load_rwkv_config(entry: ModelEntry):
    if entry.builder == "sam21_base_student":
        cfg = load_config(os.path.join(REPO_ROOT, "ext/sam2/configs/sam2.1/sam2.1_vcr.yaml"))
        base_cfg = OmegaConf.load(entry.config_path)
        cfg.model = base_cfg.model
        cfg._config_path = "sam2.1/sam2.1_hiera_t512.yaml"
        cfg.ckpt_path = entry.checkpoint_dir
        cfg.logging.filename = "sam2.1_hiera_t512_base.log"
        cfg.init.ignore_prefixes = []
        return cfg
    return load_config(entry.config_path)


def initialize_rwkv_sam2_hydra():
    config_dir = os.path.join(REPO_ROOT, "ext", "sam2", "configs")
    if not os.path.isdir(config_dir):
        raise FileNotFoundError(config_dir)
    GlobalHydra.instance().clear()
    initialize_config_dir(config_dir=config_dir, version_base="1.2")
    print("Hydra config dir for RWKV SAM2:", config_dir, flush=True)


def load_rwkv_model(entry: ModelEntry):
    ckpt_path = pick_checkpoint(entry)
    if ckpt_path is None:
        raise FileNotFoundError(f"No checkpoint yet for {entry.name}: {entry.checkpoint_dir}")
    cfg = load_rwkv_config(entry)
    cfg.training.device = str(DEVICE)
    initialize_rwkv_sam2_hydra()
    model = build_student_predictor(cfg).to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location="cpu")
    sd = unwrap_state_dict(ckpt)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    model.eval()
    return {"entry": entry, "model": model, "config": cfg, "checkpoint": ckpt_path, "missing": len(missing), "unexpected": len(unexpected)}


def load_oxford_model(entry: ModelEntry):
    for p in [OXFORD_REPO, OXFORD_SITE]:
        if p and os.path.isdir(p) and p not in sys.path:
            sys.path.insert(0, p)
    if not os.path.isfile(OXFORD_SAM_CKPT):
        raise FileNotFoundError(OXFORD_SAM_CKPT)
    if not os.path.isfile(OXFORD_MED_PRETRAIN):
        raise FileNotFoundError(OXFORD_MED_PRETRAIN)
    GlobalHydra.instance().clear()
    from sam2_train.build_sam import build_sam2_video_predictor
    hydra_overrides_extra = [
        f"model.image_size={TARGET_HW}",
        "model.memory_attention.layer.self_attention.feat_sizes=[16,16]",
        "model.memory_attention.layer.cross_attention.feat_sizes=[16,16]",
    ]
    predictor = build_sam2_video_predictor(
        config_file=entry.config_path,
        ckpt_path=OXFORD_SAM_CKPT,
        device=str(DEVICE),
        mode="train",
        hydra_overrides_extra=hydra_overrides_extra,
    )
    if hasattr(predictor, "fill_hole_area"):
        predictor.fill_hole_area = 0
    if hasattr(getattr(predictor, "model", None), "fill_hole_area"):
        predictor.model.fill_hole_area = 0
    target = predictor.model if hasattr(predictor, "model") else predictor
    ckpt = torch.load(OXFORD_MED_PRETRAIN, map_location="cpu")
    sd = unwrap_state_dict(ckpt)
    if any(str(k).startswith("module.") for k in sd.keys()):
        sd = {str(k).replace("module.", "", 1): v for k, v in sd.items()}
    missing, unexpected = target.load_state_dict(sd, strict=False)
    target.to(DEVICE).eval()
    predictor.eval()
    return {"entry": entry, "model": target, "predictor": predictor, "checkpoint": OXFORD_MED_PRETRAIN, "missing": len(missing), "unexpected": len(unexpected)}


def load_uoft_medsam2(entry: ModelEntry):
    for p in [UOFT_MEDSAM2_BASE, UOFT_MEDSAM2_SITE]:
        if p and os.path.isdir(p) and p not in sys.path:
            sys.path.insert(0, p)
    if not os.path.isfile(UOFT_MEDSAM2_CKPT):
        raise FileNotFoundError(UOFT_MEDSAM2_CKPT)
    config_dir = os.path.join(UOFT_MEDSAM2_BASE, "sam2", "configs")
    if not os.path.isdir(config_dir):
        raise FileNotFoundError(config_dir)
    GlobalHydra.instance().clear()
    initialize_config_dir(config_dir=config_dir, version_base=None)
    from sam2.build_sam import build_sam2
    model = build_sam2(entry.config_path)
    ckpt = torch.load(UOFT_MEDSAM2_CKPT, map_location="cpu")
    sd = unwrap_state_dict(ckpt)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    model.to(DEVICE).eval()
    return {"entry": entry, "model": model, "checkpoint": UOFT_MEDSAM2_CKPT, "missing": len(missing), "unexpected": len(unexpected)}


def load_model(entry: ModelEntry):
    if entry.builder in ("rwkv_student", "sam21_base_student"):
        return load_rwkv_model(entry)
    if entry.builder == "oxford_video_predictor":
        return load_oxford_model(entry)
    if entry.builder == "uoft_medsam2_build_sam2":
        return load_uoft_medsam2(entry)
    raise ValueError(f"Unknown builder: {entry.builder}")


In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1, 3, 1, 1)


def normalize_img_0_1(img_bchw):
    return (img_bchw - IMAGENET_MEAN.to(img_bchw.device)) / IMAGENET_STD.to(img_bchw.device)


def external_input_to_01(img_t3hw):
    x = ensure_image_t3hw(img_t3hw).float()
    if EXTERNAL_INPUT_PREPROCESS != "auto01":
        return x
    finite = torch.isfinite(x)
    if not bool(finite.any()):
        return x
    vals = x[finite]
    mn = float(vals.min().item())
    mx = float(vals.max().item())
    if mn >= 0.0 and mx <= 1.0:
        return x.clamp(0, 1)
    if mn >= 0.0 and mx <= 255.0:
        return (x / 255.0).clamp(0, 1)
    mn_t = vals.min()
    mx_t = vals.max()
    return ((x - mn_t) / (mx_t - mn_t + 1e-6)).clamp(0, 1)


def external_input_preprocess_note(img_t3hw):
    x = ensure_image_t3hw(img_t3hw).float()
    finite = torch.isfinite(x)
    if EXTERNAL_INPUT_PREPROCESS != "auto01":
        return str(EXTERNAL_INPUT_PREPROCESS)
    if not bool(finite.any()):
        return "auto01_nonfinite"
    vals = x[finite]
    mn = float(vals.min().item())
    mx = float(vals.max().item())
    if mn >= 0.0 and mx <= 1.0:
        return "auto01_already_0_1"
    if mn >= 0.0 and mx <= 255.0:
        return "auto01_div255"
    return "auto01_minmax"


def cuda_bf16_autocast_enabled():
    return bool(torch.cuda.is_available() and DEVICE.type == "cuda")


def forward_direct_sam_prompt(model, imgs_t3hw_01, prompt_plan, normalize_imagenet=False):
    model.eval()
    imgs = ensure_image_t3hw(imgs_t3hw_01).to(DEVICE, non_blocking=True).float()
    if normalize_imagenet:
        imgs_in = normalize_img_0_1(imgs)
    else:
        imgs_in = imgs
    T, _, H, W = imgs.shape
    with torch.inference_mode():
        backbone_out = model.forward_image(imgs_in)
        _, vision_feats, vision_pos_embeds, feat_sizes = model._prepare_backbone_features(backbone_out)
        feats = [feat.permute(1, 2, 0).reshape(T, -1, *sz) for feat, sz in zip(vision_feats[::-1], feat_sizes[::-1])][::-1]
        image_embed = feats[-1]
        hires_feats = feats[:-1]
        mode = prompt_plan["effective_prompt_mode"]
        if mode == "box":
            boxes = torch.as_tensor(prompt_plan["boxes"], dtype=torch.float32, device=DEVICE).view(T, 2, 2)
            sparse, dense = model.sam_prompt_encoder(points=None, boxes=boxes, masks=None)
        elif mode == "mask":
            masks = torch.as_tensor(prompt_plan["mask_prompts"], dtype=torch.float32, device=DEVICE).view(T, 1, H, W)
            sparse, dense = model.sam_prompt_encoder(points=None, boxes=None, masks=masks)
        else:
            raise ValueError(mode)
        if dense.shape[-2:] != image_embed.shape[-2:]:
            dense = F.interpolate(dense, size=image_embed.shape[-2:], mode="bilinear", align_corners=False)
        image_pe = model.sam_prompt_encoder.get_dense_pe()
        if image_pe.shape[-2:] != image_embed.shape[-2:]:
            image_pe = F.interpolate(image_pe, size=image_embed.shape[-2:], mode="bilinear", align_corners=False)
        image_pe = image_pe.to(image_embed.dtype)
        lowres_logits, _, _, _ = model.sam_mask_decoder(
            image_embeddings=image_embed,
            image_pe=image_pe,
            sparse_prompt_embeddings=sparse,
            dense_prompt_embeddings=dense,
            multimask_output=False,
            repeat_image=False,
            high_res_features=hires_feats,
        )
        logits = F.interpolate(lowres_logits, size=(H, W), mode="bilinear", align_corners=False)
        return logits[:, 0].detach().cpu().numpy().astype(np.float32)


def forward_video_predictor(predictor, imgs_t3hw_01, prompt_plan):
    predictor.eval()
    imgs = ensure_image_t3hw(imgs_t3hw_01).to(DEVICE, non_blocking=True).float()
    T, _, H, W = imgs.shape
    mode = prompt_plan["effective_prompt_mode"]
    autocast_enabled = cuda_bf16_autocast_enabled()
    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=autocast_enabled):
        if hasattr(predictor, "init_state_from_tensor"):
            state = predictor.init_state_from_tensor(imgs_tensor=imgs, offload_video_to_cpu=False, offload_state_to_cpu=False, mode="eval")
            add_box = getattr(predictor, "add_new_points_or_box")
            add_mask = getattr(predictor, "add_new_mask")
            propagate = getattr(predictor, "propagate_in_video")
            for t in prompt_plan["prompt_frames"]:
                if mode == "box":
                    box = torch.as_tensor(prompt_plan["boxes"][t], dtype=torch.float32, device=DEVICE)
                    empty_points = torch.empty((1, 0, 2), dtype=torch.float32, device=DEVICE)
                    empty_labels = torch.empty((1, 0), dtype=torch.int32, device=DEVICE)
                    add_box(state, int(t), 1, points=empty_points, labels=empty_labels, box=box, clear_old_points=True, normalize_coords=True)
                elif mode == "mask":
                    mask = torch.as_tensor(prompt_plan["mask_prompts"][t] > 0, dtype=torch.bool, device=DEVICE)
                    add_mask(state, int(t), 1, mask=mask)
                else:
                    raise ValueError(mode)
            logits = torch.zeros((T, H, W), device=DEVICE, dtype=torch.float32)
            for frame_idx, obj_ids, video_res_masks in propagate(state, start_frame_idx=0, max_frame_num_to_track=T, reverse=False):
                logits[int(frame_idx)] = video_res_masks[0, 0].float()
            return logits.detach().cpu().numpy().astype(np.float32)
        if hasattr(predictor, "train_init_state"):
            state = predictor.train_init_state(imgs_tensor=imgs)
            for t in prompt_plan["prompt_frames"]:
                if mode == "box":
                    box = torch.as_tensor(prompt_plan["boxes"][t], dtype=torch.float32, device=DEVICE)
                    predictor.train_add_new_bbox(state, int(t), 0, box)
                elif mode == "mask":
                    mask = torch.as_tensor(prompt_plan["mask_prompts"][t] > 0, dtype=torch.bool, device=DEVICE)
                    predictor.train_add_new_mask(state, int(t), 0, mask)
                else:
                    raise ValueError(mode)
            logits = torch.zeros((T, H, W), device=DEVICE, dtype=torch.float32)
            for out_fid, out_obj_ids, out_mask_logits in predictor.train_propagate_in_video(state, start_frame_idx=0):
                logits[int(out_fid)] = out_mask_logits[0, 0].float()
            try:
                predictor.reset_state(state)
            except Exception:
                pass
            return logits.detach().cpu().numpy().astype(np.float32)
    raise RuntimeError(f"Unsupported predictor API: {type(predictor)}")


def predict_logits(model_bundle, case, prompt_plan):
    entry = model_bundle["entry"]
    imgs = case["image"]
    if entry.builder == "uoft_medsam2_build_sam2":
        imgs01 = imgs if case.get("external_input_preprocessed", False) else external_input_to_01(imgs)
        return forward_direct_sam_prompt(model_bundle["model"], imgs01, prompt_plan, normalize_imagenet=True)
    if entry.builder == "oxford_video_predictor":
        imgs01 = imgs if case.get("external_input_preprocessed", False) else external_input_to_01(imgs)
        return forward_direct_sam_prompt(model_bundle["model"], imgs01, prompt_plan, normalize_imagenet=False)
    return forward_video_predictor(model_bundle["model"], imgs, prompt_plan)


def parameter_count(model_bundle):
    model = model_bundle.get("model", None)
    if model is None or not hasattr(model, "parameters"):
        return float("nan")
    return int(sum(p.numel() for p in model.parameters()))




def supported_prompt_modes_for_model(model_bundle):
    entry = model_bundle["entry"]
    external_builders = {"oxford_video_predictor", "uoft_medsam2_build_sam2"}
    if entry.builder in external_builders and not EVALUATE_UNSUPPORTED_EXTERNAL_PROMPTS:
        return list(EXTERNAL_BASELINE_PROMPT_MODES)
    return list(PROMPT_MODES)


def prompt_protocol_note(model_bundle):
    entry = model_bundle["entry"]
    if entry.builder in {"oxford_video_predictor", "uoft_medsam2_build_sam2"}:
        if EVALUATE_UNSUPPORTED_EXTERNAL_PROMPTS:
            return "external_exploratory_all_prompts"
        return "external_box_only_from_validation_notebooks"
    return "native_prompt_protocol"


def timed_predict(model_bundle, case, prompt_plan):
    if torch.cuda.is_available():
        if BENCHMARK_CLEAR_CUDA_CACHE_EVERY_RUN:
            torch.cuda.empty_cache()
        if BENCHMARK_RESET_CUDA_PEAK_EVERY_RUN:
            torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    logits = predict_logits(model_bundle, case, prompt_plan)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t1 = time.perf_counter()
    peak_alloc = torch.cuda.max_memory_allocated() if torch.cuda.is_available() else 0
    peak_reserved = torch.cuda.max_memory_reserved() if torch.cuda.is_available() else 0
    elapsed = float(t1 - t0)
    frames = int(logits.shape[0])
    return logits, {
        "wall_time_sec": elapsed,
        "cuda_time_sec": elapsed,
        "peak_cuda_mem_allocated_mb": peak_alloc / (1024**2),
        "peak_cuda_mem_reserved_mb": peak_reserved / (1024**2),
        "frames_per_sec": frames / elapsed if elapsed > 0 else float("nan"),
        "cases_per_sec": 1.0 / elapsed if elapsed > 0 else float("nan"),
        "n_frames": frames,
        "param_count": parameter_count(model_bundle),
        "prediction_reused": 0.0,
    }


In [ ]:
def _hd95_norm_text(value):
    if value is None:
        return ""
    try:
        if pd is not None and pd.isna(value):
            return ""
    except Exception:
        pass
    return str(value).strip()


def _hd95_int_or_none(value):
    if value is None:
        return None
    try:
        if pd is not None and pd.isna(value):
            return None
    except Exception:
        pass
    try:
        return int(float(value))
    except Exception:
        return None


def _hd95_dataset_filter_sets():
    dataset_allow = {_hd95_norm_text(x) for x in (HD95_DATASET_ALLOWLIST or [])}
    dataset_allow = {x for x in dataset_allow if x}
    index_allow = set()
    for x in (HD95_DATASET_INDEX_ALLOWLIST or []):
        idx = _hd95_int_or_none(x)
        if idx is not None:
            index_allow.add(idx)
    return dataset_allow, index_allow


def infer_hd95_active_model_names():
    if HD95_MODELS is not None:
        return set(map(str, HD95_MODELS))

    names = set()
    dataset_allow, index_allow = _hd95_dataset_filter_sets()
    needed = {"model", "status", "dim", "dataset", "dataset_index", "prompt_mode"}

    if pd is not None and Path(HD95_SOURCE_CSV).exists():
        with tqdm(total=1, desc="Read HD95 source model list", unit="file") as pbar:
            header = pd.read_csv(HD95_SOURCE_CSV, nrows=0)
            usecols = [c for c in header.columns if c in needed]
            frame = pd.read_csv(HD95_SOURCE_CSV, usecols=usecols, low_memory=False)
            pbar.update(1)
        for _, row in tqdm(frame.iterrows(), total=len(frame), desc="Infer active HD95 models", unit="row"):
            if _hd95_norm_text(row.get("status")).lower() != "ok":
                continue
            if HD95_ONLY_DIM is not None and _hd95_int_or_none(row.get("dim")) != int(HD95_ONLY_DIM):
                continue
            if dataset_allow and _hd95_norm_text(row.get("dataset")) not in dataset_allow:
                continue
            idx = _hd95_int_or_none(row.get("dataset_index"))
            if index_allow and idx not in index_allow:
                continue
            model = _hd95_norm_text(row.get("model"))
            if model:
                names.add(model)
        return names or None

    if not Path(HD95_SOURCE_JSONL).exists():
        return None
    with open(HD95_SOURCE_JSONL, "r", encoding="utf-8-sig") as f:
        for line in tqdm(f, desc="Infer active HD95 models", unit="row"):
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except Exception:
                continue
            if _hd95_norm_text(row.get("status")).lower() != "ok":
                continue
            if HD95_ONLY_DIM is not None and _hd95_int_or_none(row.get("dim")) != int(HD95_ONLY_DIM):
                continue
            if dataset_allow and _hd95_norm_text(row.get("dataset")) not in dataset_allow:
                continue
            idx = _hd95_int_or_none(row.get("dataset_index"))
            if index_allow and idx not in index_allow:
                continue
            model = _hd95_norm_text(row.get("model"))
            if model:
                names.add(model)
    return names or None


HD95_ACTIVE_MODEL_NAMES = infer_hd95_active_model_names()
if HD95_ACTIVE_MODEL_NAMES is not None:
    before = len(MODEL_REGISTRY)
    MODEL_REGISTRY = [entry for entry in MODEL_REGISTRY if entry.name in HD95_ACTIVE_MODEL_NAMES]
    print("HD95 active models:", sorted(HD95_ACTIVE_MODEL_NAMES))
    print(f"Filtered MODEL_REGISTRY: {before} -> {len(MODEL_REGISTRY)}")
else:
    print("HD95 active models: all registry entries")


In [ ]:
def get_smoke_case():
    ds = benchmark_loader.dataset
    by_dim = first_cases_by_dim(benchmark_loader, 1)
    idx = None
    for dim in [3, 2]:
        if by_dim.get(dim):
            idx = by_dim[dim][0]
            break
    if idx is None:
        idx = 0
    print("Loading smoke case index:", idx, flush=True)
    return sample_to_case(ds[idx], dataset_index=idx)


def preflight_registry(registry):
    rows = []
    loaded = {}
    smoke = get_smoke_case()
    for entry in tqdm(registry, desc="Preflight models", unit="model"):
        row = asdict(entry).copy()
        resolved_checkpoint = pick_checkpoint(entry)
        row.update({
            "config_found": bool((entry.config_path and os.path.exists(entry.config_path)) or entry.family in ("oxford", "uoft")),
            "checkpoint_found": bool(resolved_checkpoint),
            "resolved_checkpoint": resolved_checkpoint or "",
            "checkpoint_debug": checkpoint_debug(entry),
            "cuda_available": bool(torch.cuda.is_available()),
            "build_ok": False,
            "mini_inference_ok": False,
            "error": "",
        })
        print(f"Preflight {entry.name}: checkpoint_found={row['checkpoint_found']} resolved={resolved_checkpoint}", flush=True)
        if not entry.enabled:
            row["status"] = "disabled"
            rows.append(row)
            print(f"Preflight {entry.name}: disabled", flush=True)
            continue
        if not row["checkpoint_found"]:
            row["status"] = "pending"
            rows.append(row)
            print(f"Preflight {entry.name}: pending checkpoint", flush=True)
            continue
        try:
            print(f"Preflight {entry.name}: loading model...", flush=True)
            bundle = load_model(entry)
            row["build_ok"] = True
            plan = prompt_plan_for_case(smoke, "box")
            print(f"Preflight {entry.name}: running mini inference...", flush=True)
            logits, eff = timed_predict(bundle, smoke, plan)
            row["mini_inference_ok"] = bool(logits.shape == smoke["gt_thw"].shape)
            row["status"] = "available" if row["mini_inference_ok"] else "shape_failed"
            row.update({k: eff[k] for k in ["wall_time_sec", "peak_cuda_mem_allocated_mb", "param_count"]})
            loaded[entry.name] = bundle
            print(f"Preflight {entry.name}: {row['status']} in {eff['wall_time_sec']:.2f}s", flush=True)
        except Exception as e:
            row["status"] = "failed"
            row["error"] = repr(e)
            print(f"Preflight {entry.name}: failed: {e!r}", flush=True)
            traceback.print_exc(limit=2)
        rows.append(row)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
    return rows, loaded

preflight_rows, LOADED_MODELS = preflight_registry(MODEL_REGISTRY)
write_table(preflight_rows, OUT_DIR / "preflight.csv")
if pd is not None:
    display(pd.DataFrame(preflight_rows))
else:
    preflight_rows


## Load Source Rows


In [ ]:
def prompt_result_cache_key(plan):
    frames = tuple(int(t) for t in plan.get("prompt_frames", []))
    return (str(plan.get("effective_prompt_mode")), frames, str(plan.get("prompt_source", "generated")))


def _hd95_is_missing(value):
    if value is None:
        return True
    try:
        if pd is not None and pd.isna(value):
            return True
    except Exception:
        pass
    try:
        return bool(math.isnan(float(value)))
    except Exception:
        return False


def _hd95_jsonable(value):
    if _hd95_is_missing(value):
        return None
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if torch.is_tensor(value):
        if value.ndim == 0:
            return value.detach().cpu().item()
        return value.detach().cpu().tolist()
    return value


def _hd95_json_default(value):
    value = _hd95_jsonable(value)
    try:
        json.dumps(value)
        return value
    except Exception:
        return str(value)


def _hd95_norm_text(value):
    if _hd95_is_missing(value):
        return ""
    return str(value).strip()


def _hd95_int_or_none(value):
    if _hd95_is_missing(value):
        return None
    try:
        return int(float(value))
    except Exception:
        return None


def _hd95_float_or_nan(value):
    if _hd95_is_missing(value):
        return float("nan")
    try:
        return float(value)
    except Exception:
        return float("nan")


def _hd95_dataset_filter_sets():
    dataset_allow = {_hd95_norm_text(x) for x in (HD95_DATASET_ALLOWLIST or [])}
    dataset_allow = {x for x in dataset_allow if x}
    index_allow = set()
    for x in (HD95_DATASET_INDEX_ALLOWLIST or []):
        idx = _hd95_int_or_none(x)
        if idx is not None:
            index_allow.add(idx)
    return dataset_allow, index_allow


def validate_hd95_subset_controls():
    dataset_allow, index_allow = _hd95_dataset_filter_sets()
    has_filter = bool(dataset_allow) or bool(index_allow)
    if bool(HD95_REQUIRE_DATASET_FILTER) and not has_filter and HD95_MAX_CASES is None:
        raise RuntimeError(
            "HD95_REQUIRE_DATASET_FILTER is True, but HD95_DATASET_ALLOWLIST and "
            "HD95_DATASET_INDEX_ALLOWLIST are empty. Fill one of them with the paper "
            "datasets, or set HD95_MAX_CASES for a smoke run, before running HD95."
        )


def read_jsonl(path):
    rows = []
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    with open(path, "r", encoding="utf-8-sig") as f:
        for line in tqdm(f, desc=f"Read {path.name}", unit="row"):
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def read_hd95_source_rows():
    if pd is not None and Path(HD95_SOURCE_CSV).exists():
        with tqdm(total=1, desc="Read repaired final CSV", unit="file") as pbar:
            frame = pd.read_csv(HD95_SOURCE_CSV, low_memory=False)
            pbar.update(1)
        rows = []
        for raw in tqdm(frame.to_dict("records"), desc="Normalize source rows", unit="row"):
            rows.append({str(k): _hd95_jsonable(v) for k, v in raw.items()})
        return rows, Path(HD95_SOURCE_CSV), "csv"
    rows = read_jsonl(HD95_SOURCE_JSONL)
    rows = [{str(k): _hd95_jsonable(v) for k, v in row.items()} for row in rows]
    return rows, Path(HD95_SOURCE_JSONL), "jsonl"


def row_key(row):
    idx = _hd95_int_or_none(row.get("dataset_index"))
    if idx is None:
        raise ValueError(f"Missing dataset_index for row key: {row}")
    return (str(row.get("model")), str(row.get("prompt_mode")), int(idx))


def _completed_hd95_row_is_usable(row):
    if _hd95_norm_text(row.get("status")).lower() != "ok":
        return False
    if _hd95_int_or_none(row.get("dataset_index")) is None:
        return False
    return "hd95_3d" in row


def load_completed_hd95_keys(path):
    path = Path(path)
    if not path.exists():
        return set(), []
    rows = read_jsonl(path)
    keys = set()
    for r in rows:
        if _completed_hd95_row_is_usable(r):
            try:
                keys.add(row_key(r))
            except Exception:
                pass
    return keys, rows


def filter_source_rows(rows):
    dataset_allow, index_allow = _hd95_dataset_filter_sets()
    model_allow = set(map(str, HD95_MODELS)) if HD95_MODELS is not None else None
    prompt_allow = set(map(str, HD95_PROMPT_MODES)) if HD95_PROMPT_MODES is not None else None
    out = []
    skipped = Counter()
    for r in tqdm(rows, desc="Filter HD95 source rows", unit="row"):
        if _hd95_norm_text(r.get("status")).lower() != "ok":
            skipped["status"] += 1
            continue
        idx = _hd95_int_or_none(r.get("dataset_index"))
        if idx is None:
            skipped["missing_index"] += 1
            continue
        if HD95_ONLY_DIM is not None and _hd95_int_or_none(r.get("dim")) != int(HD95_ONLY_DIM):
            skipped["dim"] += 1
            continue
        if model_allow is not None and str(r.get("model")) not in model_allow:
            skipped["model"] += 1
            continue
        if prompt_allow is not None and str(r.get("prompt_mode")) not in prompt_allow:
            skipped["prompt_mode"] += 1
            continue
        if dataset_allow and _hd95_norm_text(r.get("dataset")) not in dataset_allow:
            skipped["dataset"] += 1
            continue
        if index_allow and idx not in index_allow:
            skipped["dataset_index"] += 1
            continue
        out.append(r)
    print("HD95 source filter skipped:", dict(skipped))
    return out


validate_hd95_subset_controls()
source_rows_all, HD95_SOURCE_ROWS_PATH, HD95_SOURCE_ROWS_FORMAT = read_hd95_source_rows()
source_rows = filter_source_rows(source_rows_all)
completed_keys, completed_hd95_rows = load_completed_hd95_keys(HD95_OUTPUT_JSONL)
source_rows = [r for r in tqdm(source_rows, desc="Drop completed HD95 rows", unit="row") if row_key(r) not in completed_keys]
if HD95_MAX_CASES is not None:
    keep_cases = sorted({int(r["dataset_index"]) for r in source_rows})[:int(HD95_MAX_CASES)]
    keep_cases = set(keep_cases)
    source_rows = [r for r in tqdm(source_rows, desc="Apply HD95_MAX_CASES", unit="row") if int(r["dataset_index"]) in keep_cases]

rows_by_model_case = defaultdict(list)
for r in tqdm(source_rows, desc="Group HD95 rows by model/case", unit="row"):
    rows_by_model_case[(str(r["model"]), int(r["dataset_index"]))].append(r)

print("source rows all:", len(source_rows_all), f"({HD95_SOURCE_ROWS_FORMAT}: {HD95_SOURCE_ROWS_PATH})")
print("3D/source rows pending:", len(source_rows))
print("completed hd95 rows:", len(completed_hd95_rows))
print("model/case groups:", len(rows_by_model_case))
print("source HD95 columns ignored:", HD95_SOURCE_HD95_COLUMNS_ARE_IGNORED)

if pd is not None and source_rows:
    _preview_cols = [c for c in ["dataset", "model", "prompt_mode", "dim"] if c in source_rows[0]]
    display(
        pd.DataFrame(source_rows)
        .groupby(_preview_cols, dropna=False)
        .size()
        .reset_index(name="rows")
        .sort_values(_preview_cols)
        .head(80)
    )


## GPU HD95 Helpers


In [ ]:
try:
    import cupy as cp
    from cupyx.scipy.ndimage import distance_transform_edt as cupy_distance_transform_edt
    from cupyx.scipy.ndimage import binary_erosion as cupy_binary_erosion
except Exception as e:
    cp = None
    cupy_distance_transform_edt = None
    cupy_binary_erosion = None
    print("CuPy HD95 unavailable; falling back to SciPy:", type(e).__name__, e)


def crop_slices_for_union(masks):
    coords = []
    for m in masks:
        nz = np.argwhere(np.asarray(m).astype(bool))
        if nz.size:
            coords.append(nz)
    if not coords:
        shape = np.asarray(masks[0]).shape
        return tuple(slice(0, s) for s in shape)
    all_coords = np.concatenate(coords, axis=0)
    lo = all_coords.min(axis=0)
    hi = all_coords.max(axis=0) + 1
    return tuple(slice(int(a), int(b)) for a, b in zip(lo, hi))


def surface_distances_gpu(pred_crop, gt_crop, gt_cache=None, spacing=None):
    if cp is None or not HD95_USE_GPU:
        return surface_distances_scipy(pred_crop, gt_crop, gt_cache=None, spacing=spacing)
    if spacing is None:
        spacing = (1.0,) * pred_crop.ndim
    pred_np = np.asarray(pred_crop).astype(bool)
    gt_np = np.asarray(gt_crop).astype(bool)
    if pred_np.shape != gt_np.shape:
        raise ValueError((pred_np.shape, gt_np.shape))
    if np.array_equal(pred_np, gt_np):
        return 0.0, gt_cache, 0.0, "cupy_equal_short_circuit"
    pred_any = bool(pred_np.any())
    gt_any = bool(gt_np.any())
    if not pred_any and not gt_any:
        return 0.0, gt_cache, 0.0, "cupy_empty"
    if pred_any != gt_any:
        return float("nan"), gt_cache, 0.0, "cupy_one_empty"

    t0 = time.perf_counter()
    if gt_cache is None:
        gm = cp.asarray(gt_np, dtype=cp.bool_)
        se = cp.ones((3,) * gm.ndim, dtype=cp.bool_)
        gt_surface = cp.logical_xor(gm, cupy_binary_erosion(gm, structure=se, border_value=0))
        dt_to_gt = cupy_distance_transform_edt(cp.logical_not(gt_surface), sampling=spacing)
        gt_cache = {"gt_surface": gt_surface, "dt_to_gt": dt_to_gt, "cache_sec": time.perf_counter() - t0}
    pm = cp.asarray(pred_np, dtype=cp.bool_)
    se = cp.ones((3,) * pm.ndim, dtype=cp.bool_)
    pred_surface = cp.logical_xor(pm, cupy_binary_erosion(pm, structure=se, border_value=0))
    dt_to_pred = cupy_distance_transform_edt(cp.logical_not(pred_surface), sampling=spacing)
    d_to_gt = gt_cache["dt_to_gt"][pred_surface]
    d_to_pred = dt_to_pred[gt_cache["gt_surface"]]
    if d_to_gt.size == 0 and d_to_pred.size == 0:
        value = 0.0
    elif d_to_gt.size == 0:
        value = float(cp.asnumpy(cp.percentile(d_to_pred.astype(cp.float32), 95)))
    elif d_to_pred.size == 0:
        value = float(cp.asnumpy(cp.percentile(d_to_gt.astype(cp.float32), 95)))
    else:
        value = float(cp.asnumpy(cp.percentile(cp.concatenate([d_to_gt, d_to_pred]).astype(cp.float32), 95)))
    cp.get_default_memory_pool().free_all_blocks()
    return value, gt_cache, time.perf_counter() - t0, "cupy"


def surface_distances_scipy(pred_crop, gt_crop, gt_cache=None, spacing=None):
    pred = np.asarray(pred_crop).astype(bool)
    gt = np.asarray(gt_crop).astype(bool)
    if spacing is None:
        spacing = (1.0,) * pred.ndim
    if np.array_equal(pred, gt):
        return 0.0, gt_cache, 0.0, "scipy_equal_short_circuit"
    pred_any = bool(pred.any())
    gt_any = bool(gt.any())
    if not pred_any and not gt_any:
        return 0.0, gt_cache, 0.0, "scipy_empty"
    if pred_any != gt_any:
        return float("nan"), gt_cache, 0.0, "scipy_one_empty"
    t0 = time.perf_counter()
    if gt_cache is None:
        gt_surface = np.logical_xor(gt, binary_erosion(gt))
        dt_to_gt = distance_transform_edt(~gt_surface, sampling=spacing)
        gt_cache = {"gt_surface": gt_surface, "dt_to_gt": dt_to_gt, "cache_sec": time.perf_counter() - t0}
    pred_surface = np.logical_xor(pred, binary_erosion(pred))
    dt_to_pred = distance_transform_edt(~pred_surface, sampling=spacing)
    distances = np.concatenate([gt_cache["dt_to_gt"][pred_surface], dt_to_pred[gt_cache["gt_surface"]]]).astype(np.float32)
    value = float(np.percentile(distances, 95)) if distances.size else 0.0
    return value, gt_cache, time.perf_counter() - t0, "scipy"


def full_volume_hd95_for_predictions(preds_by_cache_key, gt):
    masks = [gt] + list(preds_by_cache_key.values())
    crop = crop_slices_for_union(masks)
    gt_crop = np.asarray(gt[crop]).astype(bool)
    out = {}
    gt_cache = None
    for cache_key, pred in preds_by_cache_key.items():
        pred_crop = np.asarray(pred[crop]).astype(bool)
        value, gt_cache, elapsed, backend = surface_distances_gpu(pred_crop, gt_crop, gt_cache=gt_cache)
        out[cache_key] = {
            "hd95_3d": value,
            "hd95_backend": backend,
            "hd95_3d_time_sec": float(elapsed),
            "gt_dt_cache_sec": float(gt_cache.get("cache_sec", 0.0)) if isinstance(gt_cache, dict) else 0.0,
            "hd95_crop_shape": "x".join(map(str, gt_crop.shape)),
        }
    return out

## Run HD95 Pass


In [ ]:
def _hd95_clean_tag(value):
    text = str(value)
    return "".join(ch if ch.isalnum() else "_" for ch in text).strip("_") or "unknown"


def _hd95_markdown_table(frame, max_rows=30):
    if frame is None or len(frame) == 0:
        return "_No rows._"
    shown = frame.head(max_rows).copy()
    try:
        return shown.to_markdown(index=False)
    except Exception:
        return "```text\n" + shown.to_string(index=False) + "\n```"


def _hd95_iqr(values):
    values = pd.to_numeric(values, errors="coerce").dropna()
    if values.empty:
        return float("nan")
    return float(values.quantile(0.75) - values.quantile(0.25))


def _hd95_summary_frame(frame, group_cols):
    group_cols = [c for c in group_cols if c in frame.columns]
    if frame.empty or not group_cols:
        return pd.DataFrame()
    rows = []
    grouped = frame.groupby(group_cols, dropna=False, sort=True)
    for keys, group in tqdm(grouped, total=grouped.ngroups, desc="Summarize HD95", unit="group"):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {col: key for col, key in zip(group_cols, keys)}
        ok = group["status"].fillna("").astype(str).str.lower().eq("ok") if "status" in group.columns else pd.Series(True, index=group.index)
        row["rows"] = int(len(group))
        row["ok_rows"] = int(ok.sum())
        row["failed_rows"] = int((~ok).sum())
        for metric in ["hd95_3d", "source_dice", "source_iou", "source_dice_best", "prediction_wall_time_sec", "prediction_peak_cuda_mem_allocated_mb"]:
            if metric not in group.columns:
                continue
            values = pd.to_numeric(group[metric], errors="coerce").dropna()
            row[f"{metric}_count"] = int(len(values))
            if not values.empty:
                row[f"{metric}_mean"] = float(values.mean())
                row[f"{metric}_median"] = float(values.median())
                row[f"{metric}_std"] = float(values.std(ddof=1)) if len(values) > 1 else 0.0
                row[f"{metric}_iqr"] = _hd95_iqr(values)
        rows.append(row)
    return pd.DataFrame(rows)


def _hd95_paired_difference_summary(frame):
    comparisons = [
        ("rwkv_medsam2_nodistill", "sam2_1_base"),
        ("rwkv_medsam2_distill", "sam2_1_base"),
        ("rwkv_medsam2_nodistill", "rwkv_medsam2_distill"),
        ("rwkv_medsam2_nodistill", "oxford_medical_sam2"),
        ("rwkv_medsam2_nodistill", "uoft_medsam2"),
    ]
    required = {"model", "prompt_mode", "dataset_index", "hd95_3d"}
    if frame.empty or not required.issubset(frame.columns):
        return pd.DataFrame()
    key_cols = ["dataset_index", "prompt_mode"]
    if "dataset" in frame.columns:
        key_cols.insert(0, "dataset")
    rows = []
    ok_frame = frame[frame.get("status", "ok").astype(str).str.lower().eq("ok")].copy()
    keep_cols = key_cols + ["model", "hd95_3d"]
    for candidate, baseline in tqdm(comparisons, desc="Summarize paired HD95 deltas", unit="comparison"):
        left = ok_frame.loc[ok_frame["model"].astype(str).eq(candidate), keep_cols].drop_duplicates(key_cols, keep="last")
        right = ok_frame.loc[ok_frame["model"].astype(str).eq(baseline), keep_cols].drop_duplicates(key_cols, keep="last")
        if left.empty or right.empty:
            continue
        merged = left.merge(right, on=key_cols, suffixes=("_candidate", "_baseline"))
        if merged.empty:
            continue
        for prompt_mode, group in merged.groupby("prompt_mode", dropna=False, sort=True):
            delta = pd.to_numeric(group["hd95_3d_candidate"], errors="coerce") - pd.to_numeric(group["hd95_3d_baseline"], errors="coerce")
            delta = delta.dropna()
            rows.append({
                "comparison": f"{candidate} vs {baseline}",
                "candidate_model": candidate,
                "baseline_model": baseline,
                "prompt_mode": prompt_mode,
                "matched_rows": int(len(group)),
                "delta_hd95_3d_mean": float(delta.mean()) if not delta.empty else float("nan"),
                "delta_hd95_3d_median": float(delta.median()) if not delta.empty else float("nan"),
                "delta_hd95_3d_std": float(delta.std(ddof=1)) if len(delta) > 1 else 0.0,
                "delta_hd95_3d_iqr": _hd95_iqr(delta),
                "candidate_better_pct": float((delta < 0).mean()) if not delta.empty else float("nan"),
                "baseline_better_pct": float((delta > 0).mean()) if not delta.empty else float("nan"),
                "within_1mm_pct": float((delta.abs() <= 1.0).mean()) if not delta.empty else float("nan"),
                "within_2mm_pct": float((delta.abs() <= 2.0).mean()) if not delta.empty else float("nan"),
                "within_5mm_pct": float((delta.abs() <= 5.0).mean()) if not delta.empty else float("nan"),
            })
    return pd.DataFrame(rows)


def write_hd95_packet_outputs(rows):
    if pd is None:
        print("pandas unavailable; wrote JSONL/CSV only.")
        return {}

    frame = pd.DataFrame(rows)
    for col in ["hd95_3d", "source_dice", "source_iou", "source_dice_best", "prediction_wall_time_sec", "prediction_peak_cuda_mem_allocated_mb"]:
        if col in frame.columns:
            frame[col] = pd.to_numeric(frame[col], errors="coerce")

    merge_cols = [
        "model", "prompt_mode", "effective_prompt_mode", "dataset_index", "dataset", "subdataset",
        "task_id", "task_label", "modality", "dim", "threshold", "hd95_3d", "hd95_backend",
        "hd95_3d_time_sec", "gt_dt_cache_sec", "hd95_crop_shape", "status", "error",
    ]
    merge_frame = frame[[c for c in merge_cols if c in frame.columns]].copy() if not frame.empty else pd.DataFrame(columns=merge_cols)
    model_prompt_summary = _hd95_summary_frame(frame, ["model", "prompt_mode"])
    dataset_summary = _hd95_summary_frame(frame, ["dataset", "model", "prompt_mode"])
    paired_summary = _hd95_paired_difference_summary(frame)

    manifest = {
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S %Z"),
        "out_dir": str(OUT_DIR),
        "source_rows_path": str(HD95_SOURCE_ROWS_PATH),
        "source_rows_format": HD95_SOURCE_ROWS_FORMAT,
        "source_hd95_columns_ignored": bool(HD95_SOURCE_HD95_COLUMNS_ARE_IGNORED),
        "dataset_allowlist": list(HD95_DATASET_ALLOWLIST or []),
        "dataset_index_allowlist": list(HD95_DATASET_INDEX_ALLOWLIST or []),
        "models": list(HD95_MODELS or []),
        "prompt_modes": list(HD95_PROMPT_MODES or []),
        "only_dim": HD95_ONLY_DIM,
        "max_cases": HD95_MAX_CASES,
        "rows": int(len(frame)),
        "ok_rows": int((frame.get("status", "ok").astype(str).str.lower() == "ok").sum()) if not frame.empty else 0,
        "output_files": {
            "jsonl": str(HD95_OUTPUT_JSONL),
            "csv": str(HD95_OUTPUT_CSV),
            "merge_columns": str(HD95_OUTPUT_MERGE_CSV),
            "model_prompt_summary": str(HD95_MODEL_PROMPT_SUMMARY_CSV),
            "dataset_model_prompt_summary": str(HD95_DATASET_MODEL_PROMPT_SUMMARY_CSV),
            "paired_model_difference_summary": str(HD95_PAIRED_MODEL_DIFF_SUMMARY_CSV),
            "manifest": str(HD95_MANIFEST_JSON),
            "summary_md": str(HD95_SUMMARY_MD),
            "zip": str(HD95_PACKET_ZIP),
        },
    }

    md_lines = [
        "# Full-Volume HD95 Paper-Subset Summary",
        "",
        f"- Created: {manifest['created_at']}",
        f"- Source rows: `{manifest['source_rows_path']}`",
        f"- Source HD95 columns ignored: `{manifest['source_hd95_columns_ignored']}`",
        f"- Dataset allowlist: {', '.join(manifest['dataset_allowlist']) if manifest['dataset_allowlist'] else '_none_'}",
        f"- Dataset index allowlist: {', '.join(map(str, manifest['dataset_index_allowlist'])) if manifest['dataset_index_allowlist'] else '_none_'}",
        f"- Rows: {manifest['rows']:,}",
        "",
        "## Model/Prompt Summary",
        _hd95_markdown_table(model_prompt_summary, 80),
        "",
        "## Dataset/Model/Prompt Summary",
        _hd95_markdown_table(dataset_summary, 80),
        "",
        "## Paired HD95 Summary",
        _hd95_markdown_table(paired_summary, 80),
        "",
        "## Caveat",
        "This notebook recomputes HD95 for the selected paper subset. Existing HD95 columns in the source benchmark are not used because they may be partial or incomplete.",
    ]

    outputs = [
        (HD95_OUTPUT_MERGE_CSV, merge_frame),
        (HD95_MODEL_PROMPT_SUMMARY_CSV, model_prompt_summary),
        (HD95_DATASET_MODEL_PROMPT_SUMMARY_CSV, dataset_summary),
        (HD95_PAIRED_MODEL_DIFF_SUMMARY_CSV, paired_summary),
    ]
    for path, out_frame in tqdm(outputs, desc="Write HD95 summary CSVs", unit="file"):
        path.parent.mkdir(parents=True, exist_ok=True)
        out_frame.to_csv(path, index=False)

    HD95_MANIFEST_JSON.write_text(json.dumps(manifest, indent=2, default=_hd95_json_default), encoding="utf-8")
    HD95_SUMMARY_MD.write_text("\n".join(md_lines), encoding="utf-8")

    packet_files = [
        HD95_OUTPUT_JSONL,
        HD95_OUTPUT_CSV,
        HD95_OUTPUT_MERGE_CSV,
        HD95_MODEL_PROMPT_SUMMARY_CSV,
        HD95_DATASET_MODEL_PROMPT_SUMMARY_CSV,
        HD95_PAIRED_MODEL_DIFF_SUMMARY_CSV,
        HD95_MANIFEST_JSON,
        HD95_SUMMARY_MD,
    ]
    with zipfile.ZipFile(HD95_PACKET_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in tqdm(packet_files, desc="Zip HD95 packet", unit="file"):
            if Path(path).exists():
                zf.write(path, Path(path).name)

    print("HD95 packet outputs:")
    for path in packet_files + [HD95_PACKET_ZIP]:
        print(" ", path)
    display(model_prompt_summary)
    display(paired_summary)
    return manifest


HD95_OUTPUT_JSONL.parent.mkdir(parents=True, exist_ok=True)
all_output_rows = list(completed_hd95_rows)
rows_written_since_csv = 0

with open(HD95_OUTPUT_JSONL, "a", encoding="utf-8") as out_f:
    groups = sorted(rows_by_model_case.items(), key=lambda item: (item[0][0], item[0][1]))
    with tqdm(groups, desc="Full-volume HD95", unit="case") as pbar:
        for (model_name, dataset_index), rows in pbar:
            if model_name not in LOADED_MODELS:
                print("Skipping unavailable model:", model_name)
                continue
            bundle = LOADED_MODELS[model_name]
            case = sample_to_case(benchmark_loader.dataset[int(dataset_index)], dataset_index=int(dataset_index))
            pbar.set_postfix({"model": model_name[:18], "idx": dataset_index, "dataset": case["meta"].get("dataset")})

            pred_by_cache_key = {}
            row_cache_key = {}
            eff_by_cache_key = {}
            for source_row in rows:
                prompt_mode = str(source_row["prompt_mode"])
                plan = prompt_plan_for_case(case, prompt_mode)
                cache_key = prompt_result_cache_key(plan)
                row_cache_key[row_key(source_row)] = cache_key
                if cache_key in pred_by_cache_key:
                    continue
                logits, eff = timed_predict(bundle, case, plan)
                pred_by_cache_key[cache_key] = sigmoid_np(logits) >= float(HD95_THRESHOLD)
                eff_by_cache_key[cache_key] = eff

            hd95_by_cache_key = full_volume_hd95_for_predictions(pred_by_cache_key, case["gt_thw"])
            for source_row in rows:
                cache_key = row_cache_key[row_key(source_row)]
                hd = hd95_by_cache_key[cache_key]
                eff = eff_by_cache_key.get(cache_key, {})
                out_row = {
                    "model": source_row.get("model"),
                    "prompt_mode": source_row.get("prompt_mode"),
                    "effective_prompt_mode": source_row.get("effective_prompt_mode"),
                    "dataset_index": int(source_row.get("dataset_index")),
                    "dataset": source_row.get("dataset"),
                    "subdataset": source_row.get("subdataset"),
                    "task_id": source_row.get("task_id"),
                    "task_label": source_row.get("task_label"),
                    "modality": source_row.get("modality"),
                    "dim": int(source_row.get("dim", 3)),
                    "source_dice": source_row.get("dice"),
                    "source_iou": source_row.get("iou"),
                    "source_dice_best": source_row.get("dice_best"),
                    "source_iou_best": source_row.get("iou_best"),
                    "threshold": float(HD95_THRESHOLD),
                    **hd,
                    "prediction_wall_time_sec": eff.get("wall_time_sec", float("nan")),
                    "prediction_cuda_time_sec": eff.get("cuda_time_sec", float("nan")),
                    "prediction_frames_per_sec": eff.get("frames_per_sec", float("nan")),
                    "prediction_cases_per_sec": eff.get("cases_per_sec", float("nan")),
                    "prediction_peak_cuda_mem_allocated_mb": eff.get("peak_cuda_mem_allocated_mb", float("nan")),
                    "prediction_peak_cuda_mem_reserved_mb": eff.get("peak_cuda_mem_reserved_mb", float("nan")),
                    "param_count": eff.get("param_count", float("nan")),
                    "n_frames": int(case["gt_thw"].shape[0]),
                    "status": "ok",
                    "error": "",
                }
                out_f.write(json.dumps(out_row, default=_hd95_json_default) + "\n")
                out_f.flush()
                all_output_rows.append(out_row)
                rows_written_since_csv += 1

            if rows_written_since_csv >= int(HD95_WRITE_EVERY_ROWS):
                write_table(all_output_rows, HD95_OUTPUT_CSV)
                rows_written_since_csv = 0
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

write_table(all_output_rows, HD95_OUTPUT_CSV)
print("HD95 complete:", HD95_OUTPUT_JSONL)
hd95_manifest = write_hd95_packet_outputs(all_output_rows)
if pd is not None and all_output_rows:
    display(pd.DataFrame(all_output_rows).tail(20))
